### Embeddings(임베딩)
문장, 단어 같은 것들을 벡터 좌표로 만들어서 유사성(연관성)을 비교할 수 있게 하는 것.

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from openai import OpenAI
import pandas as pd
client = OpenAI()

text = "내가 오늘 점심을..."
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=[text]
)

print(len(response.data[0].embedding))

pd.Series(response.data[0].embedding).head()

1536


0    0.034815
1    0.013469
2   -0.054651
3   -0.014867
4    0.007306
dtype: float64

In [14]:
df = pd.read_csv("fine_food_reviews_1k.csv")
df.head()

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text
0,0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...


In [15]:
# 보내기 전 토큰수 확인
import tiktoken

# 토크나이저를 가져온다
gpt5nano_encoding = tiktoken.encoding_for_model("gpt-5-nano")

# 각 리뷰 텍스트가 몇 개의 토큰인지 계산해서 새 컬럼(n_tokens)에 저장한다.
df['n_tokens'] = df['Text'].apply(lambda x : len(gpt5nano_encoding.encode(x)))

df['n_tokens'].describe()

count    1000.000000
mean       83.818000
std        71.905308
min        22.000000
25%        38.000000
50%        59.000000
75%       104.000000
max       614.000000
Name: n_tokens, dtype: float64

In [16]:
# 전체 데이터 임베딩
def texts_to_embedding(texts):
    # 전처리 과정 : 줄바꿈 문자를 공백으로 바꿔주면 성능이 조금 더 좋아진다.
    texts = [ text.replace('\n', ' ') for text in texts]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    # 결과에서 벡터 리스트만 뽑아 반환
    return [data.embedding for data in response.data]

df['embedding'] = texts_to_embedding(df['Text'].tolist())
df['embedding'].head()

0    [0.01677853614091873, -0.008555943146348, -0.0...
1    [-0.005216312129050493, 0.040469057857990265, ...
2    [0.005564768798649311, -0.012970144860446453, ...
3    [-0.016292475163936615, 0.008886804804205894, ...
4    [-0.004322985652834177, -0.06378211826086044, ...
Name: embedding, dtype: object

In [17]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [18]:
# 의미 기반 검색 구현
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 상위 7개 비슷한 text 찾기
def get_similar_texts(query_text, df, top_k=7):
    # 사용자의 검색어도 벡터로 변환
    query_vector = texts_to_embedding([query_text])[0]

    # 데이터프레임에 있는 벡터들을 계산하기 쉽게 numpy 배열로 바꿔줌
    embeddings = np.array(df['embedding'].tolist())

    # 코사인 유사도 계산
    # query_vector를 2차원 배열로 만들어줘야 해서 []로 감싼다.
    cos_sim = cosine_similarity([query_vector], embeddings)

    df['cos_sim'] = cos_sim[0]

    return df.sort_values(by='cos_sim', ascending=False)[['Text', 'cos_sim']].head(top_k)

In [23]:
# 검색 테스트
search_result = get_similar_texts("america", df)
search_result

,Text,cos_sim
778,I didn't realize this product is made in China...,0.276927
524,My receptionist wanted some of these but I cou...,0.268614
141,"Easy to travel with. I've used in Europe, I've...",0.262887
352,"Easy to travel with. I've used in Europe, I've...",0.262887
330,"Easy to travel with. I've used in Europe, I've...",0.262887
311,The Brit's have out done us. The flavor is sup...,0.224967
381,The Brit's have out done us. The flavor is sup...,0.224967
